In [1]:
print("Hello there")

Hello there


In [2]:
from pathlib import Path
import pandas as pd

#Create a data folder for the files generated by this notebook.
from pathlib import Path
DATA_DIR = Path("../data") #./data Actual level ---- ../data 2 levels up
imdb_path = DATA_DIR / "IMDB TMDB Movie Metadata Big Dataset (1M) ORIGINAL.csv"
links_path = DATA_DIR / "links ORIGINAL.csv"

# 0. Loading datasets imbd and links 
# Those two datasets are the conection between each movie and its information
imdb_df = pd.read_csv(imdb_path)
links_df = pd.read_csv(links_path)

#### ----- Joint databases ----------

# 1. The Id at IMDB is not just numbers, then the first step is to 
# standardize ID de IMDB (delete 'tt' and transform to int)

imdb_df['imdb_id_clean'] = imdb_df['imdb_id'].str.replace('tt', '').astype(float).fillna(0).astype(int)
#imdb_df.head(10)
#links_df.head(10)

# 3. Join and merge the datasets 
# ("how = inner" merge ensure consistency)
# 'imdbId' is the conection between both datasets links.csv 
movies_combined = pd.merge(imdb_df, links_df, left_on='imdb_id_clean', right_on='imdbId', how='inner')
# how='inner' hace un inner join:
# Keep just rows that has coincidence in both dataframes
# This remove the rows that dont have equivalent key in both dataframes

# # 4. Select necessary rows to design the MVP:
columns_to_keep = [
    'title', 'genres_list', 'overview', 'keywords', 'Cast_list', 
    'Director', 'vote_average', 'release_date', 'original_language', 'movieId'
]
movies_final = movies_combined[columns_to_keep]

In [ ]:
imdb_df.head(3)
links_df.head(3)
movies_combined.head(3)
movies_final.head(5)

,title,genres_list,overview,keywords,Cast_list,Director,vote_average,release_date,original_language,movieId
0,Inception,"['Action', 'Science Fiction', 'Adventure']","Cobb, a skilled thief who commits corporate es...","['rescue', 'mission', 'dream', 'airplane', 'pa...","['Tim Kelleher', 'Silvie Laguna', 'Natasha Bea...",Christopher Nolan,8.364,2010-07-15,en,79132
1,Interstellar,"['Adventure', 'Drama', 'Science Fiction']",The adventures of a group of explorers who mak...,"['rescue', 'future', 'spacecraft', 'race again...","['Jeff Hephner', 'William Devane', 'Elyes Gabe...",Christopher Nolan,8.417,2014-11-05,en,109487
2,The Dark Knight,"['Drama', 'Action', 'Crime', 'Thriller']",Batman raises the stakes in his war on crime. ...,"['joker', 'sadism', 'chaos', 'secret identity'...","['Tommy Lister Jr.', 'Edison Chen', 'Beatrice ...",Christopher Nolan,8.512,2008-07-16,en,58559
3,Avatar,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","In the 22nd century, a paraplegic Marine is di...","['future', 'society', 'culture clash', 'space ...","['Carvon Futrell', 'Joel David Moore', 'Jon Cu...",James Cameron,7.573,2009-12-15,en,72998
4,The Avengers,"['Science Fiction', 'Action', 'Adventure']",When an unexpected enemy emerges and threatens...,"['new york city', 'superhero', 'shield', 'base...","['Haneyuri', 'Nako Mizusawa', 'Marin', 'Rikako...",Joss Whedon,7.710,2012-04-25,en,89745


# Cleaning  

In [35]:
###################    NULL MANAGMENT ###########################################

# Check null cells
# movies_final['overview'].isnull().sum() # .isnull or .isna() # Per column
# print(movies_final.isnull().sum())

# If the null case happens clean it with white spaces
# movies_final ['overview'] = movies_final ['overview'].fillna('')



################### GENRE NORMALIZATION ##########################################
# Check spaces and punctuaction 
# # Check if any genres have leading/trailing spaces
movies_final['genres_list'].str.contains('^\\s|\\s$', regex=True).sum()

###----- Function to remove spaces from each item in the list 
movies_final['genres_list'] = movies_final['genres_list'].apply(
    lambda x: [genre.strip() for genre in x] if isinstance(x, list) else x
)

### ----- Check for punctuation
# movies_final['genres_list'].str.contains(f'[{string.punctuation}]', regex=True).sum()

### Remove punctuation from genre strings ----- If it is necesary after cheking 
# import string
# movies_final['genres_list'] = movies_final['genres_list'].apply(
#     lambda x: [genre.translate(str.maketrans('', '', string.punctuation)) for genre in x] 
#     if isinstance(x, list) else x
# )


### ----- Verify all genres are clean (no spaces at edges, no punctuation) ---- If is required to cleaning
# def is_clean(genres_list):
#     if not isinstance(genres_list, list):
#         return False
#     return all(
#         genre == genre.strip() and  # No leading/trailing spaces
#         not any(c in genre for c in string.punctuation)  # No punctuation
#         for genre in genres_list
#     )
# movies_final['genres_list'].apply(is_clean).sum()  # Count clean rows


#### ----- Saving the Dataframe as CSV ---
movies_final.to_csv("../data/clean_movies.csv", index=False)



In [30]:
movies_final.head(5)

,title,genres_list,overview,keywords,Cast_list,Director,vote_average,release_date,original_language,movieId
0,Inception,"['Action', 'Science Fiction', 'Adventure']","Cobb, a skilled thief who commits corporate es...","['rescue', 'mission', 'dream', 'airplane', 'pa...","['Tim Kelleher', 'Silvie Laguna', 'Natasha Bea...",Christopher Nolan,8.364,2010-07-15,en,79132
1,Interstellar,"['Adventure', 'Drama', 'Science Fiction']",The adventures of a group of explorers who mak...,"['rescue', 'future', 'spacecraft', 'race again...","['Jeff Hephner', 'William Devane', 'Elyes Gabe...",Christopher Nolan,8.417,2014-11-05,en,109487
2,The Dark Knight,"['Drama', 'Action', 'Crime', 'Thriller']",Batman raises the stakes in his war on crime. ...,"['joker', 'sadism', 'chaos', 'secret identity'...","['Tommy Lister Jr.', 'Edison Chen', 'Beatrice ...",Christopher Nolan,8.512,2008-07-16,en,58559
3,Avatar,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","In the 22nd century, a paraplegic Marine is di...","['future', 'society', 'culture clash', 'space ...","['Carvon Futrell', 'Joel David Moore', 'Jon Cu...",James Cameron,7.573,2009-12-15,en,72998
4,The Avengers,"['Science Fiction', 'Action', 'Adventure']",When an unexpected enemy emerges and threatens...,"['new york city', 'superhero', 'shield', 'base...","['Haneyuri', 'Nako Mizusawa', 'Marin', 'Rikako...",Joss Whedon,7.710,2012-04-25,en,89745


Overview and keywords can not be null, those has to be filled with zero if that happens